# 임베딩 모델 파인튜닝 (KorQuAD 1.0 + 명사구 질문 실험 버전)

강의 자료 중 `임베딩 파인튜닝` 수업때 실습하는 자료입니다.

## 1. MultipleNegativesRankingLoss를 활용한 임베딩 파인 튜닝에 대한 상세 설명

MultipleNegativesRankingLoss는 문장 임베딩 모델을 파인 튜닝하는 데 매우 효과적인 손실 함수입니다. 이 방법에 대해 더 자세히 설명해 드리겠습니다.

손실 함수(Loss function): 오차를 계산하는 함수. AI를 학습할 때는 학습 중인 AI의 예측값과 실제 정답간의 오차를 계산해서 해당 오차를 줄이는 식으로 AI 모델을 학습합니다.

### 포지티브 샘플과 네거티브 샘플

- **포지티브 샘플(Positive Sample)**: 의미적으로 관련이 있는 문장 쌍을 의미합니다.
  - 예: (문서1: "서울의 인구는?", 문서2: "서울의 인구는 약 970만 명입니다.")

- **네거티브 샘플(Negative Sample)**: 의미적으로 관련이 없거나 관련성이 낮은 문장 쌍을 의미합니다.
  - 예: (문서1: "서울의 인구는?", 문서2: "파리는 프랑스의 수도입니다.")

### 임베딩 학습의 기본 원리

임베딩 학습의 핵심 목표는 의미적으로 유사한 텍스트는 임베딩 공간에서 가깝게, 의미적으로 다른 텍스트는 멀게 위치시키는 것입니다. (실제로 의미가 비슷하면 임베딩 유사도가 높게, 실제로 의미가 비슷하지 않으면 임베딩 유사도가 낮게 임베딩 벡터 값들을 업데이트 하는 것입니다.) 이를 위해 일반적인 임베딩 학습 방법은 다음과 같은 프로세스를 따릅니다:

1. **포지티브 샘플과 네거티브 샘플 준비**: 학습 데이터로 포지티브 샘플(관련 있는 쌍)과 네거티브 샘플(관련 없는 쌍)을 모두 준비합니다.

2. **대조 학습(Contrastive Learning)**: 모델이 포지티브 샘플 쌍의 임베딩 간 거리는 가깝게, 네거티브 샘플 쌍의 임베딩 간 거리는 멀게 학습합니다.

3. **손실 함수 최적화**: 임베딩 간 유사도(보통 코사인 유사도)를 계산하고, 포지티브 쌍의 유사도는 높이고 네거티브 쌍의 유사도는 낮추는 방향으로 손실 함수를 최적화합니다.

### 전통적인 임베딩 학습 데이터셋 구성

전통적인 대조 학습에서는 하나의 앵커(anchor)에 대해 하나의 포지티브와 여러 네거티브 샘플을 **명시적으로** 준비해야 했습니다. 이는 다음과 같은 방식으로 데이터셋을 구성해야 함을 의미합니다:

1. **트리플렛(Triplet) 구성**: 각 학습 데이터는 (앵커, 포지티브, 네거티브) 형태의 트리플렛으로 구성됩니다.
   ```python
   # 전통적인 트리플렛 구성 예시
   triplets = [
       # (앵커, 포지티브, 네거티브)
       ("강아지를 기르는 방법", "반려견 양육 가이드", "고양이 사료 추천"),
       ("파이썬 코딩 튜토리얼", "파이썬 프로그래밍 기초", "자바스크립트 입문 강의"),
       # ... 수천, 수만 개의 트리플렛 필요
   ]
   ```

2. **다중 네거티브 사례**: 효과적인 학습을 위해 하나의 앵커당 여러 개의 네거티브 샘플이 필요합니다.
   ```python
   # 다중 네거티브 샘플 구성 예시
   training_data = [
       {
           "anchor": "머신러닝이란?",
           "positive": "기계학습은 데이터로부터 패턴을 찾는 AI 기술입니다.",
           "negatives": [
               "오늘 날씨가 좋네요.",
               "내일 회의는 2시에 시작합니다.",
               "이 식당의 불고기가 맛있습니다.",
               # ... 여러 개의 네거티브 샘플
           ]
       },
       # ... 수천 개의 이런 구조
   ]
   ```

3. **데이터 준비의 어려움**:
   - 각 앵커에 대한 관련 없는 문장들을 수집해야 함
   - 적절한 난이도의 네거티브 샘플을 선택해야 함 (너무 쉽거나 어려우면 학습 효과 감소)
   - 데이터셋 크기가 기하급수적으로 증가 (앵커 × 네거티브 수)
   - 대규모 말뭉치에서 효과적인 네거티브 샘플링 전략 필요

이러한 방식은 데이터 준비 과정이 복잡하고 시간이 많이 소요되는 단점이 있었으며, 양질의 네거티브 샘플을 구성하는 것은 종종 전체 학습 프로세스에서 가장 어려운 부분 중 하나였습니다.

### MultipleNegativesRankingLoss: 배치 내 네거티브 샘플링

여기서 MultipleNegativesRankingLoss의 혁신적인 접근 방식이 등장합니다. 기존 방식과 달리, 이 손실 함수는 **명시적인 네거티브 샘플을 별도로 준비할 필요가 없다**는 큰 장점이 있습니다. 대신, 현재 배치 내의 다른 샘플들을 자동으로 네거티브로 활용합니다.

이는 데이터 준비 과정을 크게 단순화하고, 학습 효율성을 높이는 핵심 요소입니다. 사용자는 포지티브 샘플만 제공하면 되며, 네거티브 샘플은 배치 내에서 자동으로 생성됩니다.

예를 들어, 배치 크기가 4인 경우:
```python
배치 = [
    (앵커 문서1, 문서1),  # 포지티브 쌍 1
    (앵커 문서2, 문서2),  # 포지티브 쌍 2
    (앵커 문서3, 문서3),  # 포지티브 쌍 3
    (앵커 문서4, 문서4)   # 포지티브 쌍 4
]
```

앵커 문서1에 대해:
- 문서1은 포지티브 샘플
- 문서2, 문서3, 문서4는 네거티브 샘플로 취급됨

마찬가지로 앵커 문서2에 대해:
- 문서2는 포지티브 샘플
- 문서1, 문서3, 문서4는 네거티브 샘플

### MultipleNegativesRankingLoss의 실제 작동 예시

다음과 같은 문장 쌍이 있다고 가정해 보겠습니다:

```python
[
    ("AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."),
    ("딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."),
    ("Python은 어디에 쓰이나요?", "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."),
    ("자연어 처리란?", "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다.")
]
```

배치 크기가 4일 때, "AI란 무엇인가?"라는 질문(앵커)에 대해:

1. **포지티브 샘플**: "AI는 인간의 지능을 모방한 기술입니다."
2. **네거티브 샘플**:
   - "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."
   - "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."
   - "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다."

### 수학적 표현

MultipleNegativesRankingLoss는 다음과 같이 계산됩니다:

```python
L = -log( exp(sim(q, p+)) / (exp(sim(q, p+)) + Σ exp(sim(q, p-))) )
```

여기서:
- q: 쿼리/앵커 임베딩
- p+: 포지티브 샘플 임베딩
- p-: 네거티브 샘플 임베딩들
- sim(): 유사도 함수 (일반적으로 코사인 유사도)

이 손실 함수는 **포지티브 쌍의 유사도를 높이고 네거티브 쌍의 유사도를 낮추어야 값이 작아지는 식이므로 이 식을 보고 오차를 줄이는 방향으로 모델을 학습**시킵니다.

### 구현 예시 (더 상세한 코드)

```python
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader
import torch

# 모델 로드
model = SentenceTransformer('distilbert-base-multilingual-cased')

# 훈련 데이터 준비
train_examples = [
    InputExample(texts=["AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."]),
    InputExample(texts=["딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."]),
    InputExample(texts=["Python은 어디에 쓰이나요?", "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."]),
    InputExample(texts=["자연어 처리란?", "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다."])
]

# 배치 크기가 클수록 성능이 향상될 수 있음
batch_size = 32  # 실제 환경에서는 16-64 범위가 일반적
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)

# MultipleNegativesRankingLoss 설정
# 온도(temperature) 파라미터를 조정하여 손실 함수의 강도 조절 가능
loss = losses.MultipleNegativesRankingLoss(model, scale=20.0)  # scale은 temperature의 역수

# 학습 설정
train_loss = losses.MultipleNegativesRankingLoss(model)
warmup_steps = int(len(train_dataloader) * 0.1)  # 전체 훈련 데이터의 10%

# 모델 학습
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': 2e-5},
    output_path='./korean-sentence-embedding-model'
)
```

### 배치 크기의 중요성

배치 크기가 클수록 더 많은 네거티브 샘플이 생성되므로 모델의 성능이 향상될 수 있습니다:

- 배치 크기가 4인 경우: 각 질문에 대해 3개의 네거티브 샘플
- 배치 크기가 32인 경우: 각 질문에 대해 31개의 네거티브 샘플
- 배치 크기가 128인 경우: 각 질문에 대해 127개의 네거티브 샘플

### 하드 네거티브 샘플링

기본적인 배치 내 네거티브 샘플링만으로도 좋은 성능을 얻을 수 있지만, 더 어려운 네거티브 샘플을 추가하면 성능을 더욱 향상시킬 수 있습니다.

#### 하드 네거티브의 이해

하드 네거티브는 명시적으로 사용자가 직접 선택하여 학습 데이터에 포함시키는 네거티브 샘플을 의미합니다. 일반적인 MultipleNegativesRankingLoss의 배치 내 네거티브 샘플링이 자동으로 선택되는 "이지 네거티브(easy negative)"인 반면, 하드 네거티브는 사용자가 의도적으로 선별하는 샘플입니다.

#### 하드 네거티브 구현 방법

공식 문서에 따르면, MultipleNegativesRankingLoss에서 하드 네거티브는 다음과 같은 형태로 데이터를 구조화하여 제공합니다:

```python
# 하드 네거티브 예제
train_examples = [
    # (앵커, 포지티브, 하드 네거티브) 형태로 제공
    InputExample(texts=["AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다.", "AI는 로봇과 같은 물리적 형태를 가진 기계입니다."]),
    InputExample(texts=["딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.", "컴퓨터가 스스로 생각하는 방법입니다."])
]
```

이 방식에서는:
- 각 `InputExample`의 첫 번째 항목은 앵커(질문)입니다.
- 두 번째 항목은 포지티브 샘플(관련 있는 응답)입니다.
- 세 번째 이후 항목들은 하드 네거티브(관련 없지만 구분하기 어려운 응답)입니다.

손실 함수는 쌍 `(a_i, p_i)`에 대해 모든 `p_j`(`j != i`)와 모든 `n_j`를 네거티브로 사용합니다. 즉, 각 앵커는 자신의 포지티브와 유사도가 높아지도록 학습되고, 다른 앵커의 포지티브와 모든 하드 네거티브와는 유사도가 낮아지도록 학습됩니다.

#### 하드 네거티브 vs 일반 네거티브

일반 네거티브(배치 내 무작위 네거티브):
- 자동으로 배치 내에서 생성됨
- 대부분 주제가 완전히 다른 무관한 문장들
- 모델이 구분하기 상대적으로 쉬움

하드 네거티브(명시적 네거티브):
- 사용자가 직접, 의도적으로 선택함
- 포지티브와 주제는 유사하나 정확한 답변이 아님
- 미묘한 의미 차이를 포함하여 모델에게 더 큰 도전이 됨

예시:
- 질문: "당뇨병의 증상은 무엇인가요?"
- 포지티브: "당뇨병의 주요 증상으로는 갈증 증가, 빈뇨, 체중 감소 등이 있습니다."
- 하드 네거티브(명시적): "저혈당의 증상으로는 현기증, 발한, 불안감 등이 있습니다." (의료 관련 주제이지만 당뇨병이 아닌 저혈당에 관한 내용)
- 일반 네거티브(암시적): "파이썬은 객체지향 프로그래밍 언어입니다." (완전히 다른 주제)

하드 네거티브 샘플을 사용하면 모델이 더 미묘한 의미 차이를 학습하게 되어 정확도가 크게 향상될 수 있습니다. 그러나 이러한 하드 네거티브를 구성하려면 도메인 지식과 추가 작업이 필요합니다.

### 실제 응용 사례

1. **검색 시스템**: 질의어와 문서 간의 의미적 관련성 학습
2. **RAG 챗봇(Retrieval-Augmented Generation)**: 사용자 질문과 관련된 정보를 데이터베이스에서 효과적으로 검색하여 응답 생성에 활용
3. **텍스트 클러스터링**: 비슷한 문장들끼리 군집화하는 데 사용
4. **추천 시스템**: 사용자 쿼리와 관련된 아이템을 매칭하는 데 활용

### 성능 최적화 팁

1. **배치 크기 증가**: GPU 메모리가 허용하는 한 배치 크기를 늘리세요.
2. **하드 네거티브 샘플 포함**: 무작위 네거티브보다 더 효과적입니다.
3. **데이터 증강**: 원본 문장의 변형(동의어 교체, 단어 순서 변경 등)을 추가하세요.
4. **학습률 조정**: 너무 크지 않은 학습률로 시작하고 필요에 따라 조정하세요.
5. **온도(temperature) 파라미터 조정**: 손실 함수의 scale 파라미터를 조절하여 학습 강도를 제어하세요.

이러한 방법으로 MultipleNegativesRankingLoss를 활용하면 적은 양의 데이터로도 효과적인 문장 임베딩 모델을 구축할 수 있습니다.


In [2]:
!pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu124 && pip install "transformers>=4.41,<4.50" "sentence-transformers==3.4.1" "accelerate>=0.26.0" datasets openai

Looking in indexes: https://download.pytorch.org/whl/cu124

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datasets import load_dataset
from openai import OpenAI
from torch.utils.data import DataLoader
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# OpenAI API 키 설정
os.environ["OPENAI_API_KEY"] = "여러분의 키 값"
client = OpenAI()

## 2. KorQuAD 1.0 데이터 로드

KorQuAD 1.0은 한국어 위키 본문을 기반으로 만들어진 QA 데이터셋입니다.
각 샘플은 `id`, `title`, `context`, `question`, `answers` 필드로 구성되어 있습니다.

기존 실습 자료에서는 PDF 파일을 다운로드 받아 PyPDF2로 텍스트를 추출한 뒤 GPT-4o를 호출해서 문서마다 질문 2개씩을 생성했지만, KorQuAD 1.0은 이미 `(질문, 문서)` 쌍으로 구성된 데이터셋이므로 질문 생성 단계가 필요 없습니다. 학습과 평가에 바로 사용할 수 있습니다.


In [3]:
train_raw = load_dataset("KorQuAD/squad_kor_v1", split="train")
val_raw = load_dataset("KorQuAD/squad_kor_v1", split="validation")

print(f"전체 train 샘플 수: {len(train_raw)}")
print(f"전체 validation 샘플 수: {len(val_raw)}")
print(f"필드: {train_raw.column_names}")

전체 train 샘플 수: 60407
전체 validation 샘플 수: 5774
필드: ['id', 'title', 'context', 'question', 'answers']


## 3. 학습/검증 데이터 샘플링

전체 데이터가 약 6만 건으로 매우 크기 때문에 일부만 샘플링해서 사용합니다.

- 학습: 3,000개 (이 중 200개는 나중에 명사구로 변환되어 **추가** 학습 샘플로 섞임)
- 검증: 500개 (이 중 200개는 명사구 버전을 따로 만들어 **별도 평가**)

KorQuAD는 하나의 context에 질문이 여러 개 달려있는 구조이므로, 중복 질문은 걸러냅니다.


In [4]:
NUM_TRAIN = 20000
NUM_VAL = 2000
NUM_NOUN_PHRASE_TRAIN = 2000
NUM_NOUN_PHRASE_VAL = 500

train_raw = train_raw.shuffle(seed=SEED)
val_raw = val_raw.shuffle(seed=SEED)

def collect_qc_pairs(dataset, n):
    """(question, context) 쌍을 n개 수집. 너무 짧은 샘플과 중복 질문은 제외."""
    pairs = []
    seen_questions = set()
    for sample in dataset:
        q = sample['question'].strip()
        c = sample['context'].strip()
        if len(q) < 5 or len(c) < 50:
            continue
        if q in seen_questions:
            continue
        seen_questions.add(q)
        pairs.append((q, c))
        if len(pairs) >= n:
            break
    return pairs

train_pairs = collect_qc_pairs(train_raw, NUM_TRAIN)
val_pairs = collect_qc_pairs(val_raw, NUM_VAL)

print(f"학습 쌍 개수: {len(train_pairs)}")
print(f"검증 쌍 개수: {len(val_pairs)}")

print("\n[학습 샘플 예시]")
print("Q:", train_pairs[0][0])
print("C (앞 200자):", train_pairs[0][1][:200], "...")

학습 쌍 개수: 20000
검증 쌍 개수: 2000

[학습 샘플 예시]
Q: 미세먼지 해결을 위해 전국으로 확대 강화된 기존의 제도는?
C (앞 200자): 9월 26일 환경부를 비롯한 12개 관계부처가 합동으로 '미세먼지 관리 종합대책'을 확정하고 발전·산업·수송·생활 등 4개 부분에서 저감 대책을 실시하는 관련 로드맵을 발표했다. 7조 2000억 원의 예산을 투입해 미세먼지 국내 배출량을 30% 감축하고 미세먼지 '나쁨' 일수를 70%까지 줄이기로 한 것인데 이를 위해 공정률 10% 미만인 석탄발전소 9기  ...


## 4. GPT-4o로 질문 → 명사구 변환

실제 RAG 검색 환경에서는 사용자가 항상 의문형("인가요?")으로 검색하지는 않습니다. 종종 명사구("~한 것", "~한 사람") 형태로 검색하기도 합니다. 이런 케이스에도 모델이 잘 작동하도록 학습 데이터와 평가 데이터 각각에 명사구 버전 200개를 만듭니다.

예시:
- 원본: "바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?"
- 명사구: "바그너가 괴테의 파우스트를 읽고나서 쓰고자 했던 것"


In [5]:
NOUN_PHRASE_PROMPT = """다음 한국어 질문을 정보 손실 없이 명사구 형태로 바꿔주세요.
결과만 한 줄로 출력하고, 다른 설명은 쓰지 마세요.

[예시]
입력: 바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?
출력: 바그너가 괴테의 파우스트를 읽고나서 쓰고자 했던 것

입력: 대한민국의 수도는 어디인가?
출력: 대한민국의 수도

입력: 교향곡 9번을 작곡한 사람은 누구인가요?
출력: 교향곡 9번을 작곡한 사람

입력: {question}
출력:"""

def to_noun_phrase(question):
    """GPT-4o로 질문을 명사구로 변환"""
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "You convert Korean interrogative questions into noun phrases."},
            {"role": "user", "content": NOUN_PHRASE_PROMPT.format(question=question)},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()

### 4-1. 학습용 명사구 변환 (앞에서 200개)

In [6]:
MAX_WORKERS = 20  # 동시 호출 수. rate limit 보고 조절 (tier에 따라 10~50)

def convert_batch_parallel(questions_with_idx, desc=""):
    """
    [(idx, question), ...] 리스트를 받아 병렬로 명사구 변환.
    {idx: noun_phrase} 딕셔너리 반환.
    """
    result_map = {}

    def _worker(item):
        idx, q = item
        try:
            return idx, to_noun_phrase(q)
        except Exception as e:
            return idx, e

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_worker, item) for item in questions_with_idx]
        for future in tqdm(as_completed(futures), total=len(futures), desc=desc):
            idx, res = future.result()
            if isinstance(res, Exception):
                print(f"[{idx}] 변환 실패: {res}")
            else:
                result_map[idx] = res

    return result_map

이 함수는 API 호출 200번을 순서대로 하나씩 돌리지 않고, 한 번에 20개씩 동시에 돌려서 빠르게 끝내는 함수입니다.
ThreadPoolExecutor(max_workers=20)이 병렬 처리의 핵심입니다. 20개의 스레드를 만들어두고, 각 스레드가 독립적으로 API를 호출합니다. 1개가 응답을 기다리는 동안 나머지 19개도 각자 응답을 기다리니까, 대기 시간이 겹쳐지면서 전체 시간이 1/20로 줄어듭니다.
executor.submit()으로 작업을 맡기고, as_completed()로 끝난 것부터 결과를 거둡니다. 끝나는 순서가 제출 순서와 달라지니까 idx를 같이 들고 다녀서 나중에 어느 질문의 결과인지 매핑합니다.

In [7]:
print(f"학습용 질문 {NUM_NOUN_PHRASE_TRAIN}개를 명사구로 변환 중 (병렬 {MAX_WORKERS}개)...")
train_items = [(i, train_pairs[i][0]) for i in range(NUM_NOUN_PHRASE_TRAIN)]
train_noun_phrase_map = convert_batch_parallel(train_items, desc="train")

print("\n[학습 명사구 변환 예시]")
for i in list(train_noun_phrase_map.keys())[:5]:
    print(f"  원본: {train_pairs[i][0]}")
    print(f"  변환: {train_noun_phrase_map[i]}")
    print()

학습용 질문 2000개를 명사구로 변환 중 (병렬 20개)...


train:   0%|          | 0/2000 [00:00<?, ?it/s]


[학습 명사구 변환 예시]
  원본: 9세기 까지 시칠리아를 다스리던 세력은 누구인가?
  변환: 9세기까지 시칠리아를 다스리던 세력

  원본: 프리드리히 대왕의 남동생의 이름은?
  변환: 프리드리히 대왕의 남동생의 이름

  원본: 그는 근로기준법상의 감독권 행사를 요청하려고 어느 곳을 찾아갔는가?
  변환: 그가 근로기준법상의 감독권 행사를 요청하려고 찾아간 곳

  원본: EXO는 2018년 어떤 K팝 가수를 대표해 공연을 했다. 어떤 무대였을까?
  변환: EXO가 2018년에 어떤 K팝 가수를 대표해 공연했던 무대

  원본: 오랜 신학 연구 끝에 프리스틀리의 신앙은 아리우스주의를 거쳐 무엇으로 정립되었는가?
  변환: 오랜 신학 연구 끝에 프리스틀리의 신앙이 아리우스주의를 거쳐 정립된 것



### 4-2. 검증용 명사구 변환 (앞에서 200개)

검증 세트는 '원본 질문 세트'와 '명사구 버전 세트' 두 가지를 따로 만듭니다. 둘 다 정답 문서는 동일합니다. 같은 대상에 대해 질문 스타일만 바꿨을 때 성능 차이를 공정하게 비교하기 위함입니다.


In [8]:
print(f"검증용 질문 {NUM_NOUN_PHRASE_VAL}개를 명사구로 변환 중 (병렬 {MAX_WORKERS}개)...")
val_items = [(i, val_pairs[i][0]) for i in range(NUM_NOUN_PHRASE_VAL)]
val_noun_phrase_map = convert_batch_parallel(val_items, desc="val")

print("\n[검증 명사구 변환 예시]")
for i in list(val_noun_phrase_map.keys())[:5]:
    print(f"  원본: {val_pairs[i][0]}")
    print(f"  변환: {val_noun_phrase_map[i]}")
    print()

검증용 질문 500개를 명사구로 변환 중 (병렬 20개)...


val:   0%|          | 0/500 [00:00<?, ?it/s]


[검증 명사구 변환 예시]
  원본: 보름달이 뜨는 주기는?
  변환: 보름달이 뜨는 주기

  원본: 2000년대 들어 시장경제의 지속적인 발전에 따라 정한 목표는 무엇인가?
  변환: 2000년대 들어 시장경제의 지속적인 발전에 따라 정한 목표

  원본: 11월 24일 김영삼이 대통령 명령으로 제정한 법은?
  변환: 11월 24일 김영삼이 대통령 명령으로 제정한 법

  원본: 1999년 개최지 선정방식에 큰 변화를 가져다준 사건은 무엇인가?
  변환: 1999년 개최지 선정방식에 큰 변화를 가져다준 사건

  원본: 중국에서 쇼와 천황이 명령한 방화, 살인, 약탈이 포함된 작전은 뭐라고 불렸나?
  변환: 중국에서 쇼와 천황이 명령한 방화, 살인, 약탈이 포함된 작전의 명칭



## 5. 학습 데이터 구성 (센텐스 트랜스포머 형식)

MultipleNegativesRankingLoss는 유사도를 높이고 싶은 문장 쌍 데이터를 준비했을 경우 사용할 수 있는 함수입니다. 활용 예시:

- 유사 문장 쌍들
- (질의어, 응답) 쌍들
- (번역하고자 하는 언어, 번역된 언어) 쌍들

이 손실 함수는 포지티브 쌍(예: (검색어, 관련 문서))이 있는 상황에서 각 배치에서 n-1개의 네거티브 문서를 자동으로 배치 내에서 샘플링하며, 일반적으로 배치 크기가 증가할수록 성능이 향상됩니다.

명사구로 변환된 200개는 원본 질문을 '대체'하지 않고 '추가'로 섞어 넣습니다. 이렇게 해야 모델이 의문형 질문과 명사구 질문 모두에 강해집니다. 최종 학습 샘플 수는 3,000 + 200 = 3,200개입니다.


In [9]:
# 저장할 데이터 묶음
data_to_save = {
    "train_pairs": train_pairs,
    "val_pairs": val_pairs,
    "train_noun_phrase_map": {str(k): v for k, v in train_noun_phrase_map.items()},
    "val_noun_phrase_map": {str(k): v for k, v in val_noun_phrase_map.items()},
}

with open("korquad_data.json", "w", encoding="utf-8") as f:
    json.dump(data_to_save, f, ensure_ascii=False, indent=2)

print("저장 완료")
print(f"  train_pairs: {len(train_pairs)}")
print(f"  val_pairs: {len(val_pairs)}")
print(f"  train_noun_phrase_map: {len(train_noun_phrase_map)}")
print(f"  val_noun_phrase_map: {len(val_noun_phrase_map)}")

저장 완료
  train_pairs: 20000
  val_pairs: 2000
  train_noun_phrase_map: 2000
  val_noun_phrase_map: 500


In [7]:
with open("korquad_data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

train_pairs = [tuple(x) for x in loaded["train_pairs"]]
val_pairs = [tuple(x) for x in loaded["val_pairs"]]
train_noun_phrase_map = {int(k): v for k, v in loaded["train_noun_phrase_map"].items()}
val_noun_phrase_map = {int(k): v for k, v in loaded["val_noun_phrase_map"].items()}

print(len(train_pairs), len(val_pairs), len(train_noun_phrase_map), len(val_noun_phrase_map))

20000 2000 2000 500


In [8]:
train_examples = []

# (1) 원본 질문 쌍 전체 추가
for q, c in train_pairs:
    train_examples.append(InputExample(texts=[q, c]))

# (2) 명사구로 변환된 200개 추가 (동일 context와 매핑)
for i, np_q in train_noun_phrase_map.items():
    _, c = train_pairs[i]
    train_examples.append(InputExample(texts=[np_q, c]))

print(f"최종 학습 examples 개수: {len(train_examples)}")
print(f"  - 원본 질문: {len(train_pairs)}")
print(f"  - 명사구 질문(추가): {len(train_noun_phrase_map)}")

# 샘플 확인
print("\n[원본 질문 예시]")
print(train_examples[0].texts[0])
print("\n[명사구 질문 예시]")
print(train_examples[len(train_pairs)].texts[0])

최종 학습 examples 개수: 22000
  - 원본 질문: 20000
  - 명사구 질문(추가): 2000

[원본 질문 예시]
미세먼지 해결을 위해 전국으로 확대 강화된 기존의 제도는?

[명사구 질문 예시]
9세기까지 시칠리아를 다스리던 세력


## 6. 데이터 로더 및 모델/손실 함수 설정

배치 크기가 5 이상이면 Colab 환경에 따라 Out of Memory 에러가 발생할 수 있으므로 배치 크기는 4로 실습해주세요. GPU 메모리가 넉넉하다면 16~32로 늘리는 것을 권장합니다.


In [9]:
BATCH_SIZE = 10
loader = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True)

# 모델 및 손실 함수 설정
model_id = "BAAI/bge-m3"
model = SentenceTransformer(model_id)
model.max_seq_length = 2048

# 손실 함수 정의: 모델이 검색어와 포지티브 문서는 가깝게, 네거티브 문서는 멀게 학습하도록 유도
loss = losses.MultipleNegativesRankingLoss(model)

## 7. 검증 데이터셋을 InformationRetrievalEvaluator 형식으로 변환

**InformationRetrievalEvaluator 데이터 구조 및 원리 완전 설명:**

InformationRetrievalEvaluator는 정보 검색 모델을 평가하는 도구로, 다음 세 가지 필수 데이터 구조를 입력받습니다.

1. **queries (딕셔너리)**: 질문 ID를 키로, 질문 텍스트를 값으로 갖는 딕셔너리
2. **corpus (딕셔너리)**: 문서 ID를 키로, 문서 텍스트를 값으로 갖는 딕셔너리
3. **relevant_docs (딕셔너리)**: 질문 ID를 키로, 해당 질문에 관련된 문서 ID들의 집합(set)을 값으로 갖는 딕셔너리

### 이번 실습에서는 검증 세트를 두 벌 만듭니다

- **세트 A**: 원본(일반) 질문 500개
- **세트 B**: 명사구 질문 200개

두 세트 모두 **같은 corpus**를 공유합니다. 검색 대상 문서가 같아야 질문 스타일 차이에 따른 성능 변화를 공정하게 측정할 수 있습니다.

**중요**: KorQuAD는 동일 context에 여러 질문이 달려 있을 수 있으므로, context 단위로 중복 제거해서 corpus를 구성하고 같은 context에는 같은 doc_id를 부여해야 relevant_docs가 정확해집니다.

InformationRetrievalEvaluator의 동작 방식:
1. 모델이 각 질문에 대해 corpus 내 모든 문서의 관련성 점수를 계산합니다.
2. 문서들을 점수 기준으로 정렬합니다.
3. 정렬된 결과에서 relevant_docs에 명시된 문서들이 얼마나 높은 순위에 있는지 평가합니다.
4. MRR(Mean Reciprocal Rank), NDCG(Normalized Discounted Cumulative Gain), Precision@k, Recall@k 등의 메트릭을 계산합니다.


In [10]:
# 공통 corpus: val_pairs 전체 문서 (중복 context 제거)
val_corpus = {}
context_to_docid = {}
for i, (_, c) in enumerate(val_pairs):
    if c not in context_to_docid:
        doc_id = f"d{len(context_to_docid)}"
        context_to_docid[c] = doc_id
        val_corpus[doc_id] = c

print(f"검증 corpus 개수 (중복 제거 후): {len(val_corpus)}")

# --- 세트 A: 원본 질문 ---
val_dataset_original = {
    'queries': {},
    'corpus': val_corpus,
    'relevant_docs': {},
}
for i, (q, c) in enumerate(val_pairs):
    qid = f"q{i}"
    val_dataset_original['queries'][qid] = q
    val_dataset_original['relevant_docs'][qid] = {context_to_docid[c]}

# --- 세트 B: 명사구 질문 ---
val_dataset_noun_phrase = {
    'queries': {},
    'corpus': val_corpus,
    'relevant_docs': {},
}
for i, np_q in val_noun_phrase_map.items():
    _, c = val_pairs[i]
    qid = f"q{i}"
    val_dataset_noun_phrase['queries'][qid] = np_q
    val_dataset_noun_phrase['relevant_docs'][qid] = {context_to_docid[c]}

print(f"세트 A (일반 질문) 쿼리 수: {len(val_dataset_original['queries'])}")
print(f"세트 B (명사구 질문) 쿼리 수: {len(val_dataset_noun_phrase['queries'])}")

# 학습 중간 평가에는 일반 질문 세트 A를 사용
evaluator = InformationRetrievalEvaluator(
    val_dataset_original['queries'],
    val_dataset_original['corpus'],
    val_dataset_original['relevant_docs'],
)

검증 corpus 개수 (중복 제거 후): 847
세트 A (일반 질문) 쿼리 수: 2000
세트 B (명사구 질문) 쿼리 수: 500


## 8. 실제 학습

In [11]:
EPOCHS = 2

# W&B(WandB, Weights and Biases) 로깅 비활성화
# W&B는 학습 과정을 실시간으로 추적하고 시각화할 수 있는 도구입니다.
# 하지만 이 코드는 W&B 기능을 사용하지 않으므로, 이를 비활성화하여 로깅을 중단합니다.
os.environ["WANDB_DISABLED"] = "true"

# 학습 초기에 학습률을 점진적으로 증가시키는 단계 수 설정
# 전체 학습 단계의 10%를 워밍업으로 사용
warmup_steps = int(len(loader) * EPOCHS * 0.1)

# 모델 학습
model.fit(
    train_objectives=[(loader, loss)],  # 학습 데이터 로더와 손실 함수 설정
    epochs=EPOCHS,                       # 총 에포크 수
    warmup_steps=warmup_steps,           # 워밍업 단계
    output_path='exp_finetune',          # 학습된 모델 저장 경로
    show_progress_bar=True,              # 학습 진행률 표시 여부
    # evaluator=evaluator,                 # 학습 중간에 평가를 수행할 도구
    # evaluation_steps=2000,             
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.043400
1000,0.039500
1500,0.033100
2000,0.033500
2500,0.017900
3000,0.010800
3500,0.011100
4000,0.012900


## 9. 평가 함수 정의

In [12]:
def evaluate(dataset_queries, dataset_corpus, dataset_relevant_docs, model_path, top_k=5, verbose=False):
    """
    SentenceTransformer 모델을 사용하여 문서 검색 모델의 성능을 평가하는 함수

    Args:
        dataset_queries (dict): 쿼리 ID를 키로, 쿼리 텍스트를 값으로 하는 딕셔너리
        dataset_corpus (dict): 문서 ID를 키로, 문서 텍스트를 값으로 하는 딕셔너리
        dataset_relevant_docs (dict): 쿼리 ID를 키로, 관련 문서 ID 집합을 값으로 하는 딕셔너리
        model_path (str): SentenceTransformer 모델의 경로 또는 이름
        top_k (int): 각 질문당 검색할 상위 문서 개수 (기본값: 5)
        verbose (bool): 상세 출력 여부 (기본값: False)
    """
    # SentenceTransformer 모델 로드
    model = SentenceTransformer(model_path)

    # 평가 결과를 저장할 리스트
    eval_results = []

    # 문서 텍스트와 ID 준비
    corpus_texts = list(dataset_corpus.values())
    corpus_ids = list(dataset_corpus.keys())

    # 문서 임베딩 계산
    if verbose:
        print(f"문서 임베딩 계산 중 ({len(corpus_texts)}개)...")
    corpus_embeddings = model.encode(
        corpus_texts, show_progress_bar=True, convert_to_numpy=True, batch_size=32
    )
    # 코사인 유사도 계산을 위해 미리 정규화 (벡터화 버전)
    corpus_norms = corpus_embeddings / (
        np.linalg.norm(corpus_embeddings, axis=1, keepdims=True) + 1e-10
    )

    # 각 질문에 대해 검색 수행 및 평가
    for query_id, query in tqdm(dataset_queries.items()):
        # 질문 임베딩 계산
        query_embedding = model.encode(query, convert_to_numpy=True)
        query_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)
        similarities = corpus_norms @ query_norm

        # 유사도가 높은 상위 k개 문서 추출
        top_indices = np.argsort(similarities)[::-1][:top_k]
        retrieved_ids = [corpus_ids[idx] for idx in top_indices]

        # 해당 질문에 대한 실제 정답 문서 ID
        expected_ids = dataset_relevant_docs.get(query_id, set())

        # 정답 문서가 검색된 문서들 안에 있는지 확인
        is_hit = any(doc_id in retrieved_ids for doc_id in expected_ids)

        # 평가 결과 저장
        eval_result = {
            'is_hit': is_hit,           # 정답 문서가 검색 결과에 포함되었는지 여부 (True/False)
            'retrieved': retrieved_ids, # 검색된 상위 k개 문서들의 ID 목록
            'expected': list(expected_ids),    # 실제 정답 문서의 ID 목록
            'query': query_id,          # 현재 평가 중인 질문의 ID
        }
        eval_results.append(eval_result)

        if verbose and is_hit:
            print(f"쿼리 '{query[:50]}...' 에 대한 검색 성공")

    return eval_results

## 10. 4가지 조합으로 평가

(원본 모델 vs 파인튜닝 모델) × (일반 질문 vs 명사구 질문) = 4가지 조합에 대해 검색 성능을 측정합니다.

### 평가 방법: Hit@5

각 질문에 대해 모델이 corpus 안의 문서 500개와 임베딩 유사도를 계산하고, 그중 **상위 5개 문서**를 뽑습니다. 이 상위 5개 안에 정답 문서가 포함되어 있으면 "적중(hit)"으로 카운트합니다.

예를 들어 질문 2000개에 대해 적중한 질문이 1918개라면 Hit@5 = 1918 / 2000 = 0.959가 됩니다.

### Hit@5를 쓰는 이유

실제 RAG(Retrieval-Augmented Generation) 시스템에서는 검색 단계에서 보통 상위 3~10개 문서를 뽑아 LLM에 함께 넘겨줍니다. 따라서 "1등을 맞추는 능력"보다 "후보군 안에 정답이 들어오는 능력"이 실전 성능을 더 잘 반영합니다.

### 이 평가로 답하고자 하는 질문

1. 파인튜닝이 일반 의문형 질문 검색 성능을 올렸는가?
2. 파인튜닝이 명사구 질문 검색 성능도 올렸는가?
3. 원본 bge-m3 대비 파인튜닝 모델이 전반적으로 더 나아졌는가?

In [13]:
combos = [
    ("original",  "BAAI/bge-m3",  "normal",     val_dataset_original),
    ("original",  "BAAI/bge-m3",  "nounphrase", val_dataset_noun_phrase),
    ("finetuned", "exp_finetune", "normal",     val_dataset_original),
    ("finetuned", "exp_finetune", "nounphrase", val_dataset_noun_phrase),
]

summary = []
for model_tag, model_path, query_tag, ds in combos:
    name = f"{model_tag}_{query_tag}"
    print(f"\n===== {name} 평가 중 =====")

    # Hit@5 계산
    results = evaluate(ds['queries'], ds['corpus'], ds['relevant_docs'], model_path)
    hit_rate = pd.DataFrame(results)['is_hit'].mean()
    print(f"{name} Hit@5: {hit_rate:.4f}")

    summary.append({
        'model': model_tag,
        'query_type': query_tag,
        'hit@5': hit_rate,
    })

print("\n===== Hit@5 요약 =====")
print(pd.DataFrame(summary))


===== original_normal 평가 중 =====


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

original_normal Hit@5: 0.9500

===== original_nounphrase 평가 중 =====


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

...

  0%|          | 0/2000 [00:00<?, ?it/s]

finetuned_normal Hit@5: 0.9590

===== finetuned_nounphrase 평가 중 =====


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

finetuned_nounphrase Hit@5: 0.9640

===== Hit@5 요약 =====
       model  query_type  hit@5
0   original      normal  0.950
1   original  nounphrase  0.956
2  finetuned      normal  0.959
3  finetuned  nounphrase  0.964


## 핵심 요약

파인튜닝이 네 가지 조합 모두에서 원본보다 약간씩 좋아졌습니다. 일반 질문은 0.950 → 0.959 (+0.9%p), 명사구 질문은 0.956 → 0.964 (+0.8%p)로 둘 다 상승 폭이 비슷합니다. 명사구 데이터를 학습에 섞은 게 두 스타일 모두에 도움이 됐다는 뜻입니다.

## 재밌는 부분

**명사구가 일반 질문보다 점수가 높습니다.** 원본 모델에서도 (0.956 > 0.950), 파인튜닝 후에도 (0.964 > 0.959) 일관되게 이렇습니다. 보통 "의문형 질문 → 문서" 검색이 더 잘 될 거라 기대하지만 실제로는 반대입니다. 이유를 추측하면, 명사구는 핵심 명사들만 남아서 노이즈가 적고 문서의 핵심 키워드와 직접 매칭되기 쉽기 때문으로 보입니다. 반면 의문형은 "~인가요?", "무엇" 같은 의문사 토큰이 임베딩에 희석 효과를 만들 수 있어요.

## 파인튜닝 개선 폭이 작은 이유

bge-m3가 원본 상태에서도 이미 0.95 이상이라 **천장 효과(ceiling effect)**가 있습니다. 파인튜닝으로 쥐어짤 수 있는 여지가 좁아요. 0.9%p 정도면 이 범위에서는 의미 있는 개선이고, 명사구 성능도 유지한 채 일반 질문이 함께 좋아졌다는 점이 중요합니다.

## 파인튜닝이 "망가뜨리지 않았다"는 의미

파인튜닝할 때 가장 큰 걱정은 특정 도메인에 특화시키면서 다른 능력이 퇴화하는 것인데, 이 결과는 그런 징후가 없습니다. 두 스타일 모두 상승했고, 명사구 쪽이 더 크게 오르지도 않았으니 **한쪽에 과적합되지 않은 균형 잡힌 파인튜닝**입니다.

## 정리

- bge-m3는 원본이 이미 매우 강력한 베이스라인
- 파인튜닝은 작지만 일관된 개선을 제공
- 명사구 증강이 두 스타일 모두에 긍정적으로 작용
- 더 큰 개선을 원하면 학습 데이터 규모를 더 키우거나(10만 건 이상), 더 약한 베이스 모델(multilingual-e5-small 등)로 시작해야 드라마틱한 개선 폭이 보일 것